# Importing Libraries


In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from pathlib import Path
import tempfile
import zipfile


Dataset Link: https://www.kaggle.com/datasets/vipoooool/new-plant-diseases-dataset


# Project Path Setup


In [ ]:
def find_project_dir():
    candidates = [
        Path.cwd(),
        Path(r"C:\Users\Sabiq Fauzil\Documents\plant prediction"),
        Path(r"C:\Users\Sabiq Fauzil\Downloads\Plant_Disease_Prediction"),
    ]
    for candidate in candidates:
        if (candidate / "trained_model.keras").exists() or (candidate / "trained_plant_disease_model.keras").exists():
            return candidate
    return Path.cwd()

PROJECT_DIR = find_project_dir()
print("Project folder:", PROJECT_DIR)


# Test set Image Processing


In [ ]:
DEFAULT_CLASS_NAMES = ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Spot', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Tomato_mosaic_virus', 'Tomato___healthy']

validation_candidates = [
    PROJECT_DIR / "valid",
    PROJECT_DIR / "New Plant Diseases Dataset(Augmented)" / "New Plant Diseases Dataset(Augmented)" / "valid",
    PROJECT_DIR / "file unzip" / "Plant_Disease_Colab_Ready" / "valid",
    Path(r"C:\Users\Sabiq Fauzil\Downloads\Plant_Disease_Prediction\valid"),
    Path(r"C:\Users\Sabiq Fauzil\Downloads\Plant_Disease_Prediction\file unzip\Plant_Disease_Colab_Ready\valid"),
]

validation_dir = next((path for path in validation_candidates if path.exists()), None)

if validation_dir is not None:
    validation_set = tf.keras.utils.image_dataset_from_directory(
        validation_dir,
        labels="inferred",
        label_mode="categorical",
        color_mode="rgb",
        batch_size=32,
        image_size=(128, 128),
        shuffle=True,
        interpolation="bilinear",
        follow_links=False,
        crop_to_aspect_ratio=False,
    )
    class_name = validation_set.class_names
    print("Validation folder:", validation_dir)
else:
    validation_set = None
    class_name = DEFAULT_CLASS_NAMES
    print("Folder valid tidak ditemukan. Notebook memakai daftar 38 kelas bawaan dari aplikasi.")

print(class_name)


# Loading Model


In [ ]:
def build_plant_disease_model():
    model = tf.keras.Sequential(name="sequential")
    model.add(tf.keras.layers.Input(shape=(128, 128, 3), name="conv2d_input"))

    model.add(tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu", name="conv2d"))
    model.add(tf.keras.layers.Conv2D(32, 3, activation="relu", name="conv2d_1"))
    model.add(tf.keras.layers.MaxPooling2D(pool_size=2, strides=2, name="max_pooling2d"))

    model.add(tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu", name="conv2d_2"))
    model.add(tf.keras.layers.Conv2D(64, 3, activation="relu", name="conv2d_3"))
    model.add(tf.keras.layers.MaxPooling2D(pool_size=2, strides=2, name="max_pooling2d_1"))

    model.add(tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu", name="conv2d_4"))
    model.add(tf.keras.layers.Conv2D(128, 3, activation="relu", name="conv2d_5"))
    model.add(tf.keras.layers.MaxPooling2D(pool_size=2, strides=2, name="max_pooling2d_2"))

    model.add(tf.keras.layers.Conv2D(256, 3, padding="same", activation="relu", name="conv2d_6"))
    model.add(tf.keras.layers.Conv2D(256, 3, activation="relu", name="conv2d_7"))
    model.add(tf.keras.layers.MaxPooling2D(pool_size=2, strides=2, name="max_pooling2d_3"))

    model.add(tf.keras.layers.Conv2D(512, 3, padding="same", activation="relu", name="conv2d_8"))
    model.add(tf.keras.layers.Conv2D(512, 3, activation="relu", name="conv2d_9"))
    model.add(tf.keras.layers.MaxPooling2D(pool_size=2, strides=2, name="max_pooling2d_4"))

    model.add(tf.keras.layers.Dropout(0.25, name="dropout"))
    model.add(tf.keras.layers.Flatten(name="flatten"))
    model.add(tf.keras.layers.Dense(1500, activation="relu", name="dense"))
    model.add(tf.keras.layers.Dropout(0.4, name="dropout_1"))
    model.add(tf.keras.layers.Dense(38, activation="softmax", name="dense_1"))
    return model


def load_legacy_keras_model_weights(model_path):
    model = build_plant_disease_model()
    with tempfile.TemporaryDirectory() as temp_dir:
        weights_path = Path(temp_dir) / "model.weights.h5"
        with zipfile.ZipFile(model_path) as archive:
            weights_path.write_bytes(archive.read("model.weights.h5"))
        model.load_weights(str(weights_path))
    return model


def load_prediction_model(model_path):
    if not zipfile.is_zipfile(model_path):
        raise ValueError(f"{model_path} bukan file Keras .keras yang valid.")
    try:
        return tf.keras.models.load_model(model_path, compile=False)
    except TypeError as error:
        print("load_model biasa gagal, memakai loader kompatibilitas Keras lama.")
        print("Detail:", error.__class__.__name__)
        return load_legacy_keras_model_weights(model_path)


model_candidates = [
    PROJECT_DIR / "trained_model.keras",
    PROJECT_DIR / "trained_plant_disease_model.keras",
    Path(r"C:\Users\Sabiq Fauzil\Downloads\Plant_Disease_Prediction\trained_model.keras"),
    Path(r"C:\Users\Sabiq Fauzil\Downloads\Plant_Disease_Prediction\trained_plant_disease_model.keras"),
]

model_path = next((path for path in model_candidates if path.exists()), None)
if model_path is None:
    raise FileNotFoundError("Model tidak ditemukan. Pastikan trained_model.keras ada di folder proyek.")

cnn = load_prediction_model(model_path)
print("Loaded model:", model_path)


# Visualising and Performing Prediction on Single Image


In [ ]:
image_candidates = [
    PROJECT_DIR / "test" / "test" / "AppleCedarRust1.JPG",
    PROJECT_DIR / "AppleCedarRust1.JPG",
    PROJECT_DIR / "home_page.jpeg",
    Path(r"C:\Users\Sabiq Fauzil\Downloads\Plant_Disease_Prediction\test\test\AppleCedarRust1.JPG"),
    Path(r"C:\Users\Sabiq Fauzil\Downloads\Plant_Disease_Prediction\home_page.jpeg"),
]

image_path = next((path for path in image_candidates if path.exists()), None)
if image_path is None:
    raise FileNotFoundError("Gambar test tidak ditemukan. Tambahkan gambar daun, lalu isi image_path dengan path gambar tersebut.")

img = tf.keras.preprocessing.image.load_img(image_path)
plt.imshow(img)
plt.title(f"Test Image: {image_path.name}")
plt.xticks([])
plt.yticks([])
plt.show()
print("Image path:", image_path)


## Testing Model


In [ ]:
image = tf.keras.preprocessing.image.load_img(image_path, target_size=(128, 128))
input_arr = tf.keras.preprocessing.image.img_to_array(image)
input_arr = np.array([input_arr])
predictions = cnn.predict(input_arr)


In [ ]:
print(predictions)


In [ ]:
result_index = int(np.argmax(predictions))
print(result_index)


In [ ]:
model_prediction = class_name[result_index]
confidence = float(np.max(predictions))

plt.imshow(img)
plt.title(f"Disease Name: {model_prediction}\nConfidence: {confidence:.2%}")
plt.xticks([])
plt.yticks([])
plt.show()

print("Prediction:", model_prediction)
print("Confidence:", f"{confidence:.2%}")
